<a href="https://colab.research.google.com/github/zh79325/jxh-ai-server/blob/main/hub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a align="center" href="https://ultralytics.com/hub" target="_blank">
<img width="1024", src="https://github.com/ultralytics/assets/raw/main/im/ultralytics-hub.png"></a>

<div align="center">

[中文](https://docs.ultralytics.com/zh/hub/) | [한국어](https://docs.ultralytics.com/ko/hub/) | [日本語](https://docs.ultralytics.com/ja/hub/) | [Русский](https://docs.ultralytics.com/ru/hub/) | [Deutsch](https://docs.ultralytics.com/de/hub/) | [Français](https://docs.ultralytics.com/fr/hub/) | [Español](https://docs.ultralytics.com/es/hub/) | [Português](https://docs.ultralytics.com/pt/hub/) | [Türkçe](https://docs.ultralytics.com/tr/hub/) | [Tiếng Việt](https://docs.ultralytics.com/vi/hub/) | [العربية](https://docs.ultralytics.com/ar/hub/)

  <a href="https://github.com/ultralytics/hub/actions/workflows/ci.yaml"><img src="https://github.com/ultralytics/hub/actions/workflows/ci.yaml/badge.svg" alt="CI CPU"></a>
  <a href="https://colab.research.google.com/github/ultralytics/hub/blob/main/hub.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

  <a href="https://ultralytics.com/discord"><img alt="Discord" src="https://img.shields.io/discord/1089800235347353640?logo=discord&logoColor=white&label=Discord&color=blue"></a>
  <a href="https://community.ultralytics.com"><img alt="Ultralytics Forums" src="https://img.shields.io/discourse/users?server=https%3A%2F%2Fcommunity.ultralytics.com&logo=discourse&label=Forums&color=blue"></a>
  <a href="https://reddit.com/r/ultralytics"><img alt="Ultralytics Reddit" src="https://img.shields.io/reddit/subreddit-subscribers/ultralytics?style=flat&logo=reddit&logoColor=white&label=Reddit&color=blue"></a>

Welcome to the [Ultralytics](https://ultralytics.com/) HUB notebook!

This notebook allows you to train Ultralytics [YOLO](https://github.com/ultralytics/ultralytics) 🚀 models using [HUB](https://hub.ultralytics.com/). Please browse the HUB <a href="https://docs.ultralytics.com/hub/">Docs</a> for details, raise an issue on <a href="https://github.com/ultralytics/hub/issues/new/choose">GitHub</a> for support, and join our <a href="https://ultralytics.com/discord">Discord</a> community for questions and discussions!
</div>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 环境初始化

Pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml) and check software and hardware.

[![PyPI - Version](https://img.shields.io/pypi/v/ultralytics?logo=pypi&logoColor=white)](https://pypi.org/project/ultralytics/) [![Downloads](https://static.pepy.tech/badge/ultralytics)](https://www.pepy.tech/projects/ultralytics) [![PyPI - Python Version](https://img.shields.io/pypi/pyversions/ultralytics?logo=python&logoColor=gold)](https://pypi.org/project/ultralytics/)

In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
%pip install ultralytics  # install
from ultralytics import YOLO, checks, hub
from ultralytics.utils.files import WorkingDirectory
from ultralytics.utils.downloads import download
import shutil
from pathlib import Path
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
checks()  # checks

current = os.path.dirname(os.path.abspath('__file__'))
parent = Path(current+'/drive/MyDrive/codes')
if  parent.exists():
  shutil.rmtree(parent)
os.makedirs(parent)
download('https://github.com/zh79325/jxh-ai-server/archive/refs/heads/main.zip',dir=parent)



Ultralytics 8.3.56 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 32.7/235.7 GB disk)


341MB [00:19, 18.8MB/s]
Unzipping /content/drive/MyDrive/codes/main.zip to /content/drive/MyDrive/codes/jxh-ai-server-main...: 100%|██████████| 65/65 [00:02<00:00, 24.74file/s]


# 下载人工打标数据集


In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import pandas as pd
from glob import glob
import shutil
import yaml
import json
import urllib
from ultralytics.utils.downloads import download
checks()  # checks
# bentchmark https://aiimg.obs.cn-east-3.myhuaweicloud.com/train/dataset/project-4-at-2024-11-13-00-59-ce04fdff.zip
# v2 https://aiimg.obs.cn-east-3.myhuaweicloud.com:443/train%2FAI%E5%95%86%E5%93%81%E5%BC%80%E5%8F%91%E6%B5%8B%E8%AF%95%E4%BC%81%E4%B8%9A%2Fjxh%2F20241128204154_Raw_82387.zip
dataset_url='https://aiimg.obs.cn-east-3.myhuaweicloud.com/labelstudio%2FAI%E5%95%86%E5%93%81%E5%BC%80%E5%8F%91%E6%B5%8B%E8%AF%95%E4%BC%81%E4%B8%9A%2Fexport%2FYOLO_OBB%2F20250102171801_Raw_74431.zip'
current = os.path.dirname(os.path.abspath('__file__'))
root = str(parent)+'/jxh-ai-server-main/jxh-ai-models'
datasset_folder=Path(root+'/ls-export')
folder=root+'/dataset-download'
datasset_download_folder =Path(folder)
if  datasset_download_folder.exists():
  shutil.rmtree(datasset_download_folder)
os.makedirs(datasset_download_folder)
download(dataset_url,datasset_download_folder)
subfolders= [f.path for f in os.scandir(folder) if f.is_dir()]
folder=subfolders[0]
datasets = root+'/yolo_datasets'
model_yaml = os.path.abspath(f'{datasets}/data.yaml')
print('\n\n\n==========\n\n')
print('datasset_download_folder=>',folder,end='\n')
txts = glob(folder + '/**/*.txt')
images = glob(folder + '/**/*.jpg')

# Create DataFrame
df = pd.DataFrame({'txt': txts, 'image': images})

if os.path.exists(datasets):
    shutil.rmtree(datasets)
# Shuffle and split data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
all_train = len(df) < 100
train_size = int(0.8 * len(df))
if all_train:
    train_size = len(df)
train_df = df.iloc[:train_size]
test_df = []
val_df = []
if all_train:
    test_df = train_df
    val_df = train_df
else:
    test_df = df.iloc[train_size:]
    val_df = test_df.sample(frac=0.5)
    test_df = test_df.drop(val_df.index)

print('building data set')
# Create directories
for split in ['train', 'test', 'val']:
    os.makedirs(f'{datasets}/{split}/images', exist_ok=True)
    os.makedirs(f'{datasets}/{split}/labels', exist_ok=True)

# Copy files to respective directories
for split, split_df in [('train', train_df), ('test', test_df), ('val', val_df)]:
    for _, row in split_df.iterrows():
        shutil.copy(row['image'], f'{datasets}/{split}/images')
        shutil.copy(row['txt'], f'{datasets}/{split}/labels')

with open(f'{folder}/notes.json', 'r') as file:
    notes = json.load(file)

labelList = []
for label in notes['categories']:
    name = urllib.parse.unquote(label['name'])
    labelList.append(name)

print("parse yaml success labels =>", labelList)

data = dict(
    train='train/images',
    val='val/images',
    test='test/images',
    nc=len(labelList),
    names=labelList
)
with open(model_yaml, 'w') as outfile:
    yaml.dump(data, outfile, default_flow_style=False)

print('finish model_yaml in=>', model_yaml)


Ultralytics 8.3.55 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 33.4/235.7 GB disk)


100%|██████████| 16.3M/16.3M [00:00<00:00, 19.3MB/s]
Unzipping /content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/dataset-download/20241231164502_Raw_34839.zip to /content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/dataset-download/20241231164502_Raw_34839...: 100%|██████████| 24/24 [00:00<00:00, 415.02file/s]






datasset_download_folder=> /content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/dataset-download/20241231164502_Raw_34839
building data set
parse yaml success labels => ['剑南春浓香型白酒500ml', '西凤友缘酒46度', '西凤华山论剑十年', '西凤六年陈酿', '谢村花雕三年陈酿9度', '五粮液白酒(52度)', '茅台金王子酒500ml53°', '泸州老窖六年陈头曲铁盒52度', '西凤(二十年华山论剑)白酒', '汾酒53%/光瓶475ml', '红星二锅头酒52度', '顺品郎（金）480ml', '太白酒50度500ML', '牛栏山二锅头42度', '西凤酒45度(精品)375ML', '西凤55度500ml（盒）', '西凤酒55度', '中国劲酒125ML', '郎酒45度100ml', '西凤(十五年)白酒', '泸州老窖头曲白酒', '宁夏红枸杞酒']
finish model_yaml in=> /content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/yolo_datasets/data.yaml


# 模型训练

In [4]:
import os
import shutil
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.font_manager import fontManager

from ultralytics import YOLO
current = os.path.dirname(os.path.abspath('__file__'))

mpl_data_path = mpl.get_data_path()
mpl_fonts_folder = mpl_data_path + '/fonts/ttf'
print("get_data_path=>", mpl.get_data_path())
if not os.path.exists(mpl_fonts_folder + '/SimHei.ttf'):
    shutil.copyfile(root + '/SimHei.ttf', mpl_fonts_folder + '/SimHei.ttf')
fontManager.addfont(mpl_fonts_folder + '/SimHei.ttf')
plt.rcParams["font.family"] = "SimHei"
plt.rcParams['font.sans-serif'] = ['SimHei']  #用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  #用来正常显示负号

#init_model=root+'/models/input/yolo11l.pt'
init_model=root+'/models/input/yolo11l-obb.pt'
prev_model=current+'/runs/obb/train2/weights/last.pt'

train_new_model=True
print('load init_model from =>',init_model)
print('load model_yaml from =>',model_yaml)
print('load prev_model from =>',prev_model)

if train_new_model :
  model = YOLO(init_model)  # build a new model from YAML
  results = model.train(data=model_yaml,batch=4, epochs=100, imgsz=1440)
else:
  model = YOLO(prev_model)
  results = model.train(resume=True)

get_data_path=> /usr/local/lib/python3.10/dist-packages/matplotlib/mpl-data
load init_model from => /content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/models/input/yolo11l-obb.pt
load model_yaml from => /content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/yolo_datasets/data.yaml
load prev_model from => /content/runs/obb/train2/weights/last.pt
Ultralytics 8.3.55 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=obb, mode=train, model=/content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/models/input/yolo11l-obb.pt, data=/content/drive/MyDrive/codes/jxh-ai-server-main/jxh-ai-models/yolo_datasets/data.yaml, epochs=100, time=None, patience=100, batch=4, imgsz=1440, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fr

100%|██████████| 22.2M/22.2M [00:00<00:00, 41.9MB/s]


Overriding model.yaml nc=80 with nc=22

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  2    173824  ultralytics.nn.modules.block.C3k2            [128, 256, 2, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  2    691712  ultralytics.nn.modules.block.C3k2            [256, 512, 2, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  2   2234368  ultralytics.nn.modules.block.C3k2            [512, 512, 2, True]           
  7                  -1  1   2360320  ultralytic

100%|██████████| 5.35M/5.35M [00:00<00:00, 370MB/s]


AMP: checks passed ✅


...

    100/100      14.5G     0.8461      2.313      1.872        224       1440: 100%|██████████| 3/3 [00:02<00:00,  1.02it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:03<00:00,  1.28s/it]

                   all         11       1030      0.338      0.398      0.395      0.328



100 epochs completed in 0.196 hours.
Optimizer stripped from runs/obb/train/weights/last.pt, 53.2MB
Optimizer stripped from runs/obb/train/weights/best.pt, 53.2MB

Validating runs/obb/train/weights/best.pt...
Ultralytics 8.3.55 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLO11l-obb summary (fused): 483 layers, 26,144,725 parameters, 0 gradients, 90.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:03<00:00,  1.14s/it]


                   all         11       1030      0.413      0.377      0.401       0.33
         剑南春浓香型白酒500ml         10         20          0          0      0.369       0.35
              西凤友缘酒46度         11         43      0.626          1      0.987      0.892
              西凤华山论剑十年         10         47      0.379      0.745      0.685      0.605
                西凤六年陈酿         10         52      0.239      0.827      0.659      0.575
            谢村花雕三年陈酿9度         11         29          1          0      0.517      0.414
            五粮液白酒(52度)         10         20          1      0.748      0.995      0.904
        茅台金王子酒500ml53°         11         36          1          0     0.0428     0.0385
        泸州老窖六年陈头曲铁盒52度         10         52      0.248      0.846      0.735      0.556
         西凤(二十年华山论剑)白酒         10         23     0.0966     0.0435      0.046     0.0392
         汾酒53%/光瓶475ml         11        154      0.441      0.636      0.582      0.426
             红星二锅头酒52

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
       print(f"GPU index: {torch.cuda.current_device()}")

if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        print(f"Number of GPUs available: {num_gpus}")
        for i in range(num_gpus):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


Using device: cuda
GPU index: 0
Number of GPUs available: 1
GPU 0: Tesla T4
